#Model D: Word TF-IDF + Neural Network / Multi-Layer Perceptron (Michelle Wang)

In [1]:
import sys
import time
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

#Data Loading & Shared Split 
Load competition data from the team's shared zip file and apply the team's standardized 8,000/2,000 train-validation split.

In [2]:
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Add project root to sys.path so data_utils can be imported
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

# Import team's shared loading and splitting functions
from data_utils import load_data, split_data

# Path to the zip archive inside the gitignored data folder
zip_path = project_dir / "data" / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip"

# Load directly from zip archive and perform shared stratified split
train, unlabeled, test = load_data(zip_path)
train_df, val_df = split_data(train)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test.shape)

Training: (8000, 3)
Validation: (2000, 3)
Test: (3000, 2)


#Baseline Model: Word TF-IDF + Multi-Layer Perceptron (MLP)
Initial baseline using word unigrams and bigrams with a single hidden layer of 64 units, ReLU activation, and default L2 regularization ($\alpha = 0.0001$).

In [3]:
model_mlp = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,       # convert all text to lowercase
        ngram_range=(1, 2),   # extract single words (unigrams) and 2-word pairs (bigrams)
        min_df=2,             # only take words appearing at least twice
        max_features=25000    # cap features for fast training
    )),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=0.0001,         # baseline L2 regularization
        batch_size=128,
        max_iter=30,
        early_stopping=True,
        random_state=42
    ))
])

# Train the baseline model on the 8000 training samples
start_time = time.perf_counter()
model_mlp.fit(train_df["text"], train_df["label"])
train_time = time.perf_counter() - start_time
print(f"Train time: {train_time:.2f} seconds")

Train time: 56.84 seconds


In [ ]:
# Predict probability of label 1 (AI-generated) using the second column
train_prob = model_mlp.predict_proba(train_df["text"])[:, 1]
val_prob = model_mlp.predict_proba(val_df["text"])[:, 1]

# Compute AUROC
train_auc = roc_auc_score(train_df["label"], train_prob)
val_auc = roc_auc_score(val_df["label"], val_prob)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

#Hyperparameter Tuning: L2 Regularization Parameter ($\alpha$) Sweep
Compare different regularization strengths ($\alpha$) using fixed TF-IDF features to optimize validation AUROC and reduce overfitting.

In [ ]:
# Transform text using the fitted baseline vectorizer
X_train_mlp = model_mlp.named_steps["tfidf"].transform(train_df["text"])
X_val_mlp = model_mlp.named_steps["tfidf"].transform(val_df["text"])
y_train = train_df["label"]
y_val = val_df["label"]

#Evaluate in the range of alpha values
alpha_values = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
grid_results = []

for v in alpha_values:
    #initialize and fit MLP with specific alpha
    model = MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=v,
        batch_size=128,
        max_iter=30,
        early_stopping=True,
        random_state=42
    )

    model.fit(X_train_mlp, y_train)

    #predict positive 1 probabilities
    train_prob = model.predict_proba(X_train_mlp)[:, 1]
    val_prob = model.predict_proba(X_val_mlp)[:, 1]

    #record AUROC metrics
    grid_results.append({
        "Alpha": v,
        "Train AUROC": roc_auc_score(y_train, train_prob),
        "Validation AUROC": roc_auc_score(y_val, val_prob)
    })

#Format comparison table
grid_df = pd.DataFrame(grid_results)
print(grid_df.round(4).to_string(index=False))

#Find optimal alpha achieving highest valudation AUROC
optimal_alpha = grid_df.loc[grid_df["Validation AUROC"].idxmax(), "Alpha"]
print(f"\nOptimal Alpha: {optimal_alpha}")

In [ ]:
#plot the curve on validation and training AUROC across alpha values
plt.figure(figsize=(9, 5))
plt.plot(grid_df["Alpha"], grid_df["Validation AUROC"], marker="o", label="Validation AUROC")
plt.plot(grid_df["Alpha"], grid_df["Train AUROC"], marker="o", linestyle="--", label="Train AUROC")
plt.xscale("log")  # use log scale because alpha values span multiple orders of magnitude
plt.xlabel(r"$\alpha$ (log scale)")
plt.ylabel("AUROC")
plt.title("Multi-Layer Perceptron: L2 Regularization Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#Retrain on Full Dataset & Final Kaggle Submission
Retrain the Multi-Layer Perceptron pipeline on all 10,000 labeled samples using the tuned regularization parameter ($\alpha$), generate positive-class test probabilities, verify submission formatting, and export the final CSV directly to Desktop.

In [ ]:
# Recreate best tuned MLP model
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=25000
    )),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=optimal_alpha,
        batch_size=128,
        max_iter=35,
        early_stopping=True,
        random_state=42
    ))
])

# Train on all 10,000 labeled ex
final_model.fit(train["text"], train["label"])

# Predict probability of AI-generated text for Kaggle test data
test_probs = final_model.predict_proba(test["text"])[:, 1]
print(f"Training examples: {len(train)}")
print(f"Test predictions:  {len(test_probs)}")

In [ ]:
# Load kaggle's sample submission directly from zip archive
with zipfile.ZipFile(zip_path) as z:
    sample_file = next(
        name for name in z.namelist()
        if name.endswith("sample_submission.csv")
        and not name.startswith("__MACOSX/")
    )
    submission = pd.read_csv(z.open(sample_file))

# Identify prediction column
prediction_cols = [col for col in submission.columns if col != "id"]
assert len(prediction_cols) == 1
prediction_col = prediction_cols[0]

# Match predicted probabilities to correct test IDs
prob_by_id = pd.Series(test_probs, index=test["id"])
submission[prediction_col] = submission["id"].map(prob_by_id)

# Verify submission before saving
assert len(submission) == len(test)
assert submission[prediction_col].notna().all()
assert submission[prediction_col].between(0, 1).all()

# Save submission file directly to Desktop
desktop_dir = Path.home() / "Desktop"
output_path = desktop_dir / "submission_neural_network_mlp.csv"

submission.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(submission.head())